In [ ]:
# =========================================================
# PARTE 1: Cálculo de vecinos N(x) y grado |N(x)| del caballo
# =========================================================

# Lista para convertir índice de columna (0-7) a letra de ajedrez (a-h)
conv = ["a", "b", "c", "d", "e", "f", "g", "h"]

# N[r][c] guardará la lista de vecinos (como tuplas de índices (fila, col))
# a los que el caballo puede moverse desde la casilla (r, c)
N = [[[] for _ in range(8)] for _ in range(8)]

# car_N[r][c] guardará el cardinal |N(x)|, es decir, cuántos vecinos tiene (r, c)
car_N = [[0 for _ in range(8)] for _ in range(8)]

# Función auxiliar SOLO para presentación: convierte índices (r, c) en base 0
# a notación de ajedrez, ej: to_chess(0, 0) -> "a1"
# Nota: se mantiene separada de N para no perder los índices numéricos,
# que se necesitan luego para construir la matriz de transición P.
def to_chess(r, c):
    return f"{conv[r]}{c+1}"

# Recorremos cada casilla (r, c) del tablero.
# Para cada una, revisamos los 8 posibles movimientos en L del caballo,
# verificando en cada caso que el destino no se salga del tablero (índices 0 a 7).
for r in range(8):
    for c in range(8):

        # Movimientos "2 filas arriba" (r-2)
        if r - 2 > -1:
            if c - 1 > -1:
                N[r][c].append((r - 2, c - 1))
                car_N[r][c] += 1
            if c + 1 < 8:
                N[r][c].append((r - 2, c + 1))
                car_N[r][c] += 1

        # Movimientos "2 filas abajo" (r+2)
        if r + 2 < 8:
            if c - 1 > -1:
                N[r][c].append((r + 2, c - 1))
                car_N[r][c] += 1
            if c + 1 < 8:
                N[r][c].append((r + 2, c + 1))
                car_N[r][c] += 1

        # Movimientos "2 columnas a la izquierda" (c-2)
        if c - 2 > -1:
            if r - 1 > -1:
                N[r][c].append((r - 1, c - 2))
                car_N[r][c] += 1
            if r + 1 < 8:
                N[r][c].append((r + 1, c - 2))
                car_N[r][c] += 1

        # Movimientos "2 columnas a la derecha" (c+2)
        if c + 2 < 8:
            if r - 1 > -1:
                N[r][c].append((r - 1, c + 2))
                car_N[r][c] += 1
            if r + 1 < 8:
                N[r][c].append((r + 1, c + 2))
                car_N[r][c] += 1


In [ ]:
# =========================================================
# PARTE 1b: Tabla con x, N(x), |N(x)|
# =========================================================

# Imprimimos la tabla pedida: casilla, sus vecinos (en notación de ajedrez)
# y la cantidad de vecinos. Usamos to_chess() solo aquí, para no alterar
# la estructura numérica de N que se reutiliza más adelante.
print(f"x\tN(x)\t|N(x)|")
for i in range(8):
    for j in range(8):
        # Casilla actual y su grado
        print(f"{conv[i]}{j + 1}\t{car_N[i][j]}", end="\t")
        # Lista de vecinos convertidos a notación de ajedrez
        for k in range(len(N[i][j])):
            print(to_chess(*N[i][j][k]), end="  ")
        print()


In [ ]:
# =========================================================
# PARTE 2: Construcción de la matriz de transición P (64x64)
# =========================================================

# P[x][y] = probabilidad de pasar del estado x al estado y en un movimiento.
# Se indexan los 64 estados con la fórmula: indice = fila*8 + columna
# (orden: a1,a2,...,a8, b1,...,b8, ..., h1,...,h8)
P = [[0 for _ in range(64)] for _ in range(64)]

# Recorremos cada casilla (r, c) y usamos los vecinos ya calculados en N,
# junto con el grado car_N, para llenar la fila correspondiente de P.
# IMPORTANTE: esto se hace en un bucle SEPARADO del anterior, porque
# car_N[r][c] debe estar completamente calculado antes de usarlo para dividir.
for r in range(8):
    for c in range(8):
        x = r * 8 + c              # índice de la casilla actual (fila de P)
        for (nr, nc) in N[r][c]:   # recorremos cada vecino ya calculado
            y = nr * 8 + nc        # índice del vecino (columna de P)
            P[x][y] = 1 / car_N[r][c]   # probabilidad uniforme entre vecinos


In [ ]:
# =========================================================
# PARTE 2b: Verificaciones y consulta de filas específicas de P
# =========================================================

# Verificación 1: cada fila debe sumar 1 (el caballo siempre se mueve a algún vecino)
print("¿Todas las filas suman 1?", all(abs(sum(fila) - 1) < 1e-9 for fila in P))

# Verificación 2: la diagonal debe ser toda cero (el caballo nunca se queda quieto)
print("¿Diagonal toda cero?", all(P[i][i] == 0 for i in range(64)))

# Función auxiliar para mostrar una fila de P en formato legible,
# solo con las probabilidades distintas de cero (para no imprimir 64 ceros)
def mostrar_fila(nombre_casilla, r, c):
    x = r * 8 + c
    print(f"\nFila de P para {nombre_casilla}:")
    for y in range(64):
        if P[x][y] != 0:
            nr, nc = divmod(y, 8)
            print(f"  P({nombre_casilla} -> {to_chess(nr, nc)}) = {P[x][y]:.4f}")

# Ejemplo pedido en el enunciado: filas de g1 y h6
mostrar_fila("g1", 6, 0)   # g -> índice 6, fila 1 -> índice 0
mostrar_fila("h6", 7, 5)   # h -> índice 7, fila 6 -> índice 5
